# 실제 golden 데이터 따라가기
생성된 `golden.npz`와 HEX 파일을 **읽어서** 한 단계씩 확인합니다. 커널은 **Python (aissd)** 를 선택하고 위에서부터 실행하세요.

표시 구간의 계산값을 저장된 값과 비교합니다. 데이터 생성·체크포인트 추론·RTL 시뮬레이션은 실행하지 않습니다. 기존 `mlp_layer_view`와 달리 **FC2·FC3 입력도 앞 레이어의 INT8 출력**입니다.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "project/03_model/src/golden_view.py").is_file())
sys.path.insert(0, str(ROOT / "project/03_model"))
from src.golden_view import GoldenRun

## 1. 확인할 실행 결과 선택
`RUN_DIR`에 비교할 시뮬레이션과 같은 golden 실행 폴더를 지정하세요. 최신 폴더를 자동으로 고르지 않습니다. 파일을 다시 생성한 경우 이 셀부터 다시 실행하세요.

In [2]:
RUN_DIR = ROOT / "project/07_runs/golden_sample_00000_20261109_012337_273666"
run = GoldenRun(RUN_DIR)
run.summary()

run = golden_sample_00000_20261109_012337_273666
sample=0 | label=7 | FP32=7 | INT8=7
golden.npz SHA-256: PASS | 표시하는 HEX 파일도 처음 읽을 때 해시 확인


## 2. 원본 이미지 일부 → 정규화 → 입력 코드
`PIXEL_ROW`, `PIXEL_COL`, `PIXEL_COUNT`로 볼 구간을 바꿉니다. count는 1~8입니다. F는 소수 비트 수이며, 복원한 값은 `정수 코드 / 2^F`입니다.

In [3]:
PIXEL_ROW = 7
PIXEL_COL = 4
PIXEL_COUNT = 8
run.pixels(row=PIXEL_ROW, col=PIXEL_COL, count=PIXEL_COUNT)

pixel[7, 4:12] = [0, 0, 84, 185, 159, 151, 60, 36]
normalized = [-0.42421 -0.42421  0.64496  1.93051  1.59958  1.49775  0.33948  0.034  ]
input8 (F5) = [-14, -14, 21, 62, 51, 48, 11, 1]


## 3. 레이어·뉴런·입력 워드 선택
`LAYER`: fc1/fc2/fc3, `NEURON`: 출력 뉴런 번호, `WORD_START`: 입력 워드 시작 번호, `WORD_COUNT`: 표시할 워드 수(1~8).

워드 하나는 입력 4개입니다. `tile = NEURON // 4`, `출력 lane = NEURON % 4`이며 weight bank도 출력 lane과 같습니다. FC3은 뉴런 10·11을 선택해 dummy 0도 확인할 수 있습니다. 설정을 바꾼 뒤에는 **이 아래 셀을 다시 실행**하세요.

In [4]:
LAYER = "fc1"
NEURON = {"fc1": 29, "fc2": 0, "fc3": 7}[LAYER]
WORD_START = {"fc1": 50, "fc2": 0, "fc3": 0}[LAYER]
WORD_COUNT = 2

step = run.select(LAYER, neuron=NEURON, word_start=WORD_START, word_count=WORD_COUNT)
step.summary()

fc1 neuron=29 → tile=7, output lane/bank=1
input word=50..51 | input index=200..207
F: input=5, weight=7, bias=9, acc=12, output=2
bias_shift=3 | output_shift=10 | ReLU=True


## 4. 실제 입력
정수 코드와 32비트 packed HEX입니다. FC1은 이미지의 input8, FC2·FC3은 앞 레이어 INT8 출력입니다. 첫 값은 `[7:0]`, 마지막 값은 `[31:24]`에 들어갑니다.

In [5]:
step.inputs()

word[50] input[200:204] = [-14, -14, 21, 62] | packed HEX=3E15F2F2
word[51] input[204:208] = [51, 48, 11, 1] | packed HEX=010B3033


## 5. 가중치
선택한 뉴런의 weight 4개씩 표시합니다. 실제 `memory/weight_bankN.hex`의 같은 주소와 대조합니다.

In [6]:
step.weights()

bank1[1422] weight[29,200:204] = [-1, -3, -3, -13] | HEX=F3FDFDFF
bank1[1423] weight[29,204:208] = [0, -6, 3, 14] | HEX=0E03FA00


## 6. 곱셈 4개
위 입력과 가중치를 signed 값으로 곱합니다. RTL 곱셈·adder tree 경로는 int32입니다.

In [7]:
step.products()

word[50] input × weight = [14, 42, -63, -806]
word[51] input × weight = [0, -288, 33, 14]


## 7. Adder tree → psum
`(mul0 + mul1) + (mul2 + mul3)`를 계산해 저장된 `psum32[tile, input_word, output_lane]`과 비교합니다. 여기서 psum은 **입력 네 곱의 합 하나**입니다.

In [8]:
step.adder_tree()

word[50] pair01=56, pair23=-869 → psum=-813 | HEX=FFFFFCD3 | NPZ 일치
word[51] pair01=-288, pair23=47 → psum=-241 | HEX=FFFFFF0F | NPZ 일치


## 8. Bias → 누산기 초기값
bias8을 부호 확장하고 소수점에 맞춰 shift합니다. `acc32[tile, 0, lane]`은 입력을 더하기 전 bias32입니다.

In [9]:
step.bias()

bias word[7] HEX=C20CEE25 | 선택 lane=1
bias8=-18 (HEX=EE) → 부호 확장 후 shift 3 → bias32=-144 (HEX=FFFFFF70)
acc[prefix=0]=-144 | NPZ 일치


## 9. 선택한 구간의 누산
`acc[k] + psum[k] = acc[k+1]`입니다. 중간 워드를 골랐다면 `acc[k]`에는 **bias와 앞선 모든 워드**가 이미 포함되어 있습니다.

In [10]:
step.accumulation()

word[50] acc[50] -6906 + psum -813 = acc[51] -7719 | HEX=FFFFE1D9 | NPZ 일치
word[51] acc[51] -7719 + psum -241 = acc[52] -7960 | HEX=FFFFE0E8 | NPZ 일치


## 10. 전체 입력 누산의 최종값
위에서 일부만 표시했어도, 최종값은 해당 뉴런의 **전체 입력**과 bias를 사용합니다. 전체 내적을 계산해 마지막 누산값과 대조합니다.

In [11]:
step.final_sum()

전체 784개 입력: bias32 + Σ(input × weight) = 14037
acc[prefix=196]=14037 | HEX=000036D5 | NPZ 일치


## 11. ReLU → 산술 우측 shift → signed 8비트 포화
FC1·FC2에 ReLU를 적용하고 FC3은 생략합니다. 음수의 우측 shift는 음의 무한대 방향으로 버립니다. 결과를 `[-128, 127]`로 제한합니다.

In [12]:
step.postprocess()

acc32=14037 → ReLU(True)=14037 → >>10=13
signed int8 포화=13 | HEX=0D | NPZ output8=13 일치
복원한 값 = 13 / 2^2 = 3.25


## 12. 출력 4개 → 버퍼 저장 워드
선택한 뉴런이 포함된 tile 전체의 출력과 실제 `expected/fcN_output.hex` 값을 확인합니다. FC1·FC3은 B, FC2는 A에 저장합니다.

In [13]:
step.store()

output neurons[28:32] = [0, 13, 15, 0]
buffer B word[7] ← HEX=000F0D00 | lane0=[7:0], lane3=[31:24]
expected/fc1_output.hex: line index=7 (편집기 줄 번호=8)


## 13. RTL에서 찾을 주소
모듈 포트의 **로컬 word address**입니다. 주소가 몇 번째 clock에 나오는지는 별도의 Control 검증 벡터가 정합니다. 이 노트북의 word 번호는 cycle 번호가 아닙니다.

In [14]:
step.addresses()

모듈 word address (TB 초기화용 byte address와 구분):
input A[50] | weight bank1[1422]
input A[51] | weight bank1[1423]
bias[7] | output B[7]


## 14. psum·누산 결과의 실제 HEX 줄
배열과 HEX 모두 **0부터 시작하는 index**로 표시합니다. 편집기의 줄 번호는 `index + 1`입니다. `psum32` 순서는 `[tile, input_word, lane]`, `acc32`는 `[tile, prefix, lane]`입니다. 여기에 표시되는 acc는 선택한 psum을 더한 **뒤**의 값입니다.

In [15]:
step.expected_hex()

word[50] fc1_psum32.hex[5689]=FFFFFCD3 | fc1_acc32.hex[5721]=FFFFE1D9
word[51] fc1_psum32.hex[5693]=FFFFFF0F | fc1_acc32.hex[5725]=FFFFE0E8


## 15. 최종 ARGMAX
마지막 출력은 signed INT8 코드로 비교합니다. class 0~9만 비교하며 dummy 10·11은 제외합니다. `expected/prediction.hex`도 대조합니다.

In [16]:
run.argmax()

class 0..9 signed code = [-62, -20, -6, -1, -5, -33, -60, 53, -17, 2]
dummy 10..11 = [0, 0] (비교 제외)
ARGMAX = 7 | label = 7 | 동률은 가장 작은 index


## 16. 임의의 메모리 구간 확인
`MEMORY_NAME`: buffer_a/b, weight_bank0~3, bias. `MEMORY_START`: word 주소, `MEMORY_COUNT`: 1~8.

`FINAL=False`는 **TB 초기화 데이터**, `FINAL=True`는 **FC3 종료 후 버퍼 전체**입니다(final은 buffer에만 사용). HEX는 왼쪽이 상위 비트, signed lanes는 lane0부터 표시합니다. 아래 TB byte 주소는 `manifest의 base + 4 × word`입니다.

In [17]:
MEMORY_NAME = "buffer_b"
MEMORY_START = 0
MEMORY_COUNT = 3
FINAL = True

run.memory(MEMORY_NAME, start=MEMORY_START, count=MEMORY_COUNT, final=FINAL)

expected/buffer_b_final.hex | FC3 종료 후
word[0] | TB byte=0x20000 | HEX=FFFAECC2 | signed lanes=[-62, -20, -6, -1]
word[1] | TB byte=0x20004 | HEX=35C4DFFB | signed lanes=[-5, -33, -60, 53]
word[2] | TB byte=0x20008 | HEX=000002EF | signed lanes=[-17, 2, 0, 0]


최종 buffer B의 word 0~2는 FC3 결과이며 나머지 주소에는 FC1 결과가 남아 있습니다. FC2가 읽는 B 입력은 `fc2_input8`/FC1 출력으로 확인하세요. 최종 버퍼만으로 중간 시점 전체를 나타낼 수는 없습니다.

실제 시뮬레이션에서는 같은 golden 실행 폴더를 사용하고, **하강 에지**에서 얻은 주소·lane·signed 정수/HEX를 위 값과 비교하세요. 이 노트북은 산술과 저장 형식 확인용이며 RTL의 지연이나 PASS 여부를 판정하지 않습니다.